In [0]:
# Define the Silver and Gold base paths
silver_base = "/Volumes/workspace/default/trucking_analytics/silver"
gold_base = "/Volumes/workspace/default/trucking_analytics/gold"

In [0]:
# Import PySpark functions required for Gold analytics
from pyspark.sql.functions import (
    col,
    sum,
    avg,
    count,
    when,
    round,
    month,
    year,
    datediff,
    coalesce
)
# Import Delta Lake functionality
from delta.tables import DeltaTable

In [0]:
# Read the Silver tables required for Gold analytics

drivers = spark.read.format("delta").load(f"{silver_base}/drivers")
trucks = spark.read.format("delta").load(f"{silver_base}/trucks")
customers = spark.read.format("delta").load(f"{silver_base}/customers")
routes = spark.read.format("delta").load(f"{silver_base}/routes")
loads = spark.read.format("delta").load(f"{silver_base}/loads")
trips = spark.read.format("delta").load(f"{silver_base}/trips")
fuel_purchases = spark.read.format("delta").load(f"{silver_base}/fuel_purchases")
maintenance_records = spark.read.format("delta").load(f"{silver_base}/maintenance_records")
delivery_events = spark.read.format("delta").load(f"{silver_base}/delivery_events")
safety_incidents = spark.read.format("delta").load(f"{silver_base}/safety_incidents")

**1. **Driver Performance:** How is each driver performing in terms of on-time delivery rate, MPG, and revenue generated per mile?**

In [0]:
# ============================================================
# BUSINESS QUESTION 1: DRIVER PERFORMANCE
# How is each driver performing in terms of on-time delivery
# rate, MPG, and revenue generated per mile?
# ============================================================

# Calculate driver MPG and revenue per mile
driver_performance_metrics = (
    trips.alias("t")
    .join(
        loads.alias("l"),
        col("t.load_id") == col("l.load_id"),
        "left"
    )
    .join(
        drivers.alias("d"),
        col("t.driver_id") == col("d.driver_id"),
        "left"
    )
    .groupBy(
        col("t.driver_id").alias("driver_id"),
        col("d.first_name").alias("first_name"),
        col("d.last_name").alias("last_name")
    )
    .agg(
        round(
            avg("t.average_mpg"),
            2
        ).alias("average_mpg"),

        round(
            sum("l.revenue") / sum("t.actual_distance_miles"),
            2
        ).alias("revenue_per_mile")
    )
)


# Calculate on-time delivery rate for each driver
driver_on_time = (
    delivery_events.alias("e")
    .join(
        trips.alias("t"),
        col("e.trip_id") == col("t.trip_id"),
        "left"
    )
    .groupBy(
        col("t.driver_id").alias("driver_id")
    )
    .agg(
        round(
            avg(
                when(
                    col("e.on_time_flag") == True,
                    1
                ).otherwise(0)
            ) * 100,
            2
        ).alias("on_time_delivery_rate")
    )
)


# Combine all driver performance metrics
driver_performance_gold = (
    driver_performance_metrics
    .join(
        driver_on_time,
        "driver_id",
        "left"
    )
    .select(
        "driver_id",
        "first_name",
        "last_name",
        "on_time_delivery_rate",
        "average_mpg",
        "revenue_per_mile"
    )
)


# Define the Gold table path
driver_performance_gold_path = f"{gold_base}/driver_performance"


# Create the Gold Delta table on the first run
if not DeltaTable.isDeltaTable(spark, driver_performance_gold_path):

    driver_performance_gold.write \
        .format("delta") \
        .save(driver_performance_gold_path)


# Update existing drivers and insert new drivers
else:

    driver_performance_table = DeltaTable.forPath(
        spark,
        driver_performance_gold_path
    )

    (
        driver_performance_table.alias("target")
        .merge(
            driver_performance_gold.alias("source"),
            "target.driver_id = source.driver_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


# Display the final Driver Performance Gold table
display(
    spark.read
    .format("delta")
    .load(driver_performance_gold_path)
    .limit(30)
)

driver_id,first_name,last_name,on_time_delivery_rate,average_mpg,revenue_per_mile
DRV00069,Patricia,Taylor,54.58,6.5,2.14
DRV00050,Robert,Johnson,56.31,6.48,2.15
DRV00097,Thomas,Rodriguez,54.69,6.48,2.12
DRV00146,Jessica,Gonzalez,54.15,6.49,2.17
DRV00071,Jessica,Johnson,55.91,6.48,2.1
DRV00098,David,Anderson,53.89,6.55,2.14
DRV00104,Richard,Hernandez,56.8,6.5,2.15
DRV00012,Robert,Moore,54.69,6.52,2.09
DRV00125,David,Smith,56.39,6.5,2.15
DRV00058,David,Miller,55.68,6.49,2.16


**2. **Route Profitability:** Which routes or lanes are most and least profitable based on revenue versus operating costs?**


In [0]:
# ============================================================
# BUSINESS QUESTION 2: ROUTE PROFITABILITY
# Which routes/lanes are most/least profitable based on
# revenue vs operating costs?
# ============================================================
# Import required PySpark functions
from pyspark.sql.functions import (
    col,
    sum,
    avg,
    count,
    when,
    round,
    month,
    year,
    datediff,
    coalesce,
    lit
)
# Calculate route revenue
route_revenue = (
    loads.alias("l")
    .join(
        routes.alias("r"),
        col("l.route_id") == col("r.route_id"),
        "left"
    )
    .groupBy(
        col("l.route_id"),
        col("r.origin_city"),
        col("r.origin_state"),
        col("r.destination_city"),
        col("r.destination_state")
    )
    .agg(
        round(sum("l.revenue"), 2).alias("total_revenue")
    )
)


# Calculate fuel cost by route
route_fuel_cost = (
    fuel_purchases.alias("f")
    .join(
        trips.alias("t"),
        col("f.trip_id") == col("t.trip_id"),
        "left"
    )
    .join(
        loads.alias("l"),
        col("t.load_id") == col("l.load_id"),
        "left"
    )
    .groupBy("l.route_id")
    .agg(
        round(sum("f.total_cost"), 2).alias("fuel_cost")
    )
)


# Calculate maintenance cost by route
route_maintenance_cost = (
    maintenance_records.alias("m")
    .join(
        trips.alias("t"),
        col("m.truck_id") == col("t.truck_id"),
        "left"
    )
    .join(
        loads.alias("l"),
        col("t.load_id") == col("l.load_id"),
        "left"
    )
    .groupBy("l.route_id")
    .agg(
        round(sum("m.total_cost"), 2).alias("maintenance_cost")
    )
)


# Combine revenue, fuel cost, and maintenance cost
route_profitability_gold = (
    route_revenue
    .join(
        route_fuel_cost,
        "route_id",
        "left"
    )
    .join(
        route_maintenance_cost,
        "route_id",
        "left"
    )

    # Replace missing costs with zero
    .withColumn(
        "fuel_cost",
        coalesce(col("fuel_cost"), lit(0))
    )
    .withColumn(
        "maintenance_cost",
        coalesce(col("maintenance_cost"), lit(0))
    )

    # Calculate total operating cost
    .withColumn(
        "total_operating_cost",
        round(
            col("fuel_cost") + col("maintenance_cost"),
            2
        )
    )

    # Calculate profit
    .withColumn(
        "profit",
        round(
            col("total_revenue") - col("total_operating_cost"),
            2
        )
    )

    # Calculate profit margin
    .withColumn(
        "profit_margin",
        round(
            (col("profit") / col("total_revenue")) * 100,
            2
        )
    )
)


# Define the Gold table path
route_profitability_gold_path = f"{gold_base}/route_profitability"


# Create the Gold Delta table on the first run
if not DeltaTable.isDeltaTable(spark, route_profitability_gold_path):

    route_profitability_gold.write \
        .format("delta") \
        .save(route_profitability_gold_path)


# Update existing routes and insert new routes
else:

    route_profitability_table = DeltaTable.forPath(
        spark,
        route_profitability_gold_path
    )

    (
        route_profitability_table.alias("target")
        .merge(
            route_profitability_gold.alias("source"),
            "target.route_id = source.route_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


# Display the final Route Profitability Gold table
display(
    spark.read
    .format("delta")
    .load(route_profitability_gold_path)
    .limit(30)
)

route_id,origin_city,origin_state,destination_city,destination_state,total_revenue,fuel_cost,maintenance_cost,total_operating_cost,profit,profit_margin
RTE00006,Chicago,IL,Charlotte,NC,2105438.99,1639811.74,6.707905367E7,6.871886541E7,-6.661342642E7,-3163.87
RTE00002,Atlanta,GA,Miami,FL,2154291.12,1663359.66,6.736751875E7,6.903087841E7,-6.687658729E7,-3104.34
RTE00037,Las Vegas,NV,New York,NY,5770647.8,1608384.69,6.815369896E7,6.976208365E7,-6.399143585E7,-1108.91
RTE00014,Phoenix,AZ,Kansas City,MO,3840447.09,1654412.93,6.878435146E7,7.043876439E7,-6.65983173E7,-1734.13
RTE00035,Las Vegas,NV,Dallas,TX,2990648.75,1589151.96,6.592833044E7,6.75174824E7,-6.452683365E7,-2157.62
RTE00047,Columbus,OH,Philadelphia,PA,1595152.57,1786126.58,7.549433186E7,7.728045844E7,-7.568530587E7,-4744.71
RTE00042,Charlotte,NC,Seattle,WA,9070885.01,1615945.96,6.651636701E7,6.813231297E7,-5.906142796E7,-651.11
RTE00020,Houston,TX,Seattle,WA,5712509.24,1650825.37,6.791873133E7,6.95695567E7,-6.385704746E7,-1117.85
RTE00058,Kansas City,MO,Indianapolis,IN,1682085.01,1588173.91,6.680371205E7,6.839188596E7,-6.670980095E7,-3965.9
RTE00030,Seattle,WA,Indianapolis,IN,7681556.33,1644240.2,6.633596294E7,6.798020314E7,-6.029864681E7,-784.98


**3. **Fleet Utilization:** How effectively is each truck/asset being utilized based on miles driven and revenue generated per asset?
**

In [0]:
# ============================================================
# BUSINESS QUESTION 3: FLEET UTILIZATION
# How effectively is each truck/asset utilized based on
# miles driven and revenue generated per asset?
# ============================================================

# Calculate truck utilization metrics from trips and loads
fleet_utilization_gold = (
    trips.alias("t")
    .join(
        loads.alias("l"),
        col("t.load_id") == col("l.load_id"),
        "left"
    )
    .join(
        trucks.alias("tr"),
        col("t.truck_id") == col("tr.truck_id"),
        "left"
    )
    .groupBy(
        col("t.truck_id").alias("truck_id"),
        col("tr.unit_number").alias("unit_number")
    )
    .agg(
        count("t.trip_id").alias("total_trips"),

        sum("t.actual_distance_miles").alias(
            "total_miles_driven"
        ),

        round(
            sum("l.revenue"),
            2
        ).alias(
            "total_revenue"
        )
    )

    # Calculate revenue generated per mile
    .withColumn(
        "revenue_per_mile",
        round(
            col("total_revenue") /
            col("total_miles_driven"),
            2
        )
    )
)


# Calculate revenue generated per truck
fleet_utilization_gold = (
    fleet_utilization_gold
    .withColumn(
        "revenue_per_trip",
        round(
            col("total_revenue") /
            col("total_trips"),
            2
        )
    )
)


# Define the Gold table path
fleet_utilization_gold_path = f"{gold_base}/fleet_utilization"


# Create the Gold Delta table on the first run
if not DeltaTable.isDeltaTable(spark, fleet_utilization_gold_path):

    fleet_utilization_gold.write \
        .format("delta") \
        .save(fleet_utilization_gold_path)


# Update existing trucks and insert new trucks
else:

    fleet_utilization_table = DeltaTable.forPath(
        spark,
        fleet_utilization_gold_path
    )

    (
        fleet_utilization_table.alias("target")
        .merge(
            fleet_utilization_gold.alias("source"),
            "target.truck_id = source.truck_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


# Display the final Fleet Utilization Gold table
display(
    spark.read
    .format("delta")
    .load(fleet_utilization_gold_path)
    .limit(30)
)

truck_id,unit_number,total_trips,total_miles_driven,total_revenue,revenue_per_mile,revenue_per_trip
TRK00049,8277,912,1268102,2701436.34,2.13,2962.1
TRK00082,3414,924,1324616,2821632.99,2.13,3053.72
TRK00013,3093,885,1258508,2668794.49,2.12,3015.59
TRK00062,7256,909,1273747,2741939.16,2.15,3016.43
TRK00113,1311,880,1265996,2697325.8,2.13,3065.14
TRK00084,2502,927,1333901,2830994.87,2.12,3053.93
TRK00043,3712,919,1321161,2844190.8,2.15,3094.88
TRK00073,2680,935,1338563,2872595.75,2.15,3072.29
TRK00015,2531,961,1386604,2979377.79,2.15,3100.29
TRK00095,7326,927,1281742,2774709.48,2.16,2993.21


**4. **Maintenance Analysis:** What is the maintenance cost per mile, and how does vehicle downtime impact fleet operations and profitability?**


In [0]:
# ============================================================
# BUSINESS QUESTION 4: MAINTENANCE ANALYSIS
# What is maintenance cost per mile, and how does downtime
# impact operations/profitability?
# ============================================================

# Calculate maintenance cost and downtime for each truck
maintenance_summary = (
    maintenance_records.alias("m")
    .groupBy(
        col("m.truck_id").alias("truck_id")
    )
    .agg(
        round(
            sum("m.total_cost"),
            2
        ).alias("total_maintenance_cost"),

        round(
            sum("m.downtime_hours"),
            2
        ).alias("total_downtime_hours"),

        count("m.maintenance_id").alias(
            "maintenance_events"
        )
    )
)


# Calculate total miles driven for each truck
truck_miles = (
    trips.alias("t")
    .groupBy(
        col("t.truck_id").alias("truck_id")
    )
    .agg(
        sum("t.actual_distance_miles").alias(
            "total_miles_driven"
        ),

        round(
            sum("t.actual_distance_miles") /
            count("t.trip_id"),
            2
        ).alias(
            "average_miles_per_trip"
        )
    )
)


# Combine maintenance and truck mileage information
maintenance_gold = (
    maintenance_summary
    .join(
        truck_miles,
        "truck_id",
        "left"
    )

    # Calculate maintenance cost per mile
    .withColumn(
        "maintenance_cost_per_mile",
        round(
            col("total_maintenance_cost") /
            col("total_miles_driven"),
            2
        )
    )

    # Calculate downtime hours per 1,000 miles
    .withColumn(
        "downtime_hours_per_1000_miles",
        round(
            (
                col("total_downtime_hours") /
                col("total_miles_driven")
            ) * 1000,
            2
        )
    )
)


# Define the Gold table path
maintenance_gold_path = f"{gold_base}/maintenance_analysis"


# Create the Gold Delta table on the first run
if not DeltaTable.isDeltaTable(spark, maintenance_gold_path):

    maintenance_gold.write \
        .format("delta") \
        .save(maintenance_gold_path)


# Update existing trucks and insert new trucks
else:

    maintenance_table = DeltaTable.forPath(
        spark,
        maintenance_gold_path
    )

    (
        maintenance_table.alias("target")
        .merge(
            maintenance_gold.alias("source"),
            "target.truck_id = source.truck_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


# Display the final Maintenance Analysis Gold table
display(
    spark.read
    .format("delta")
    .load(maintenance_gold_path)
    .limit(30)
)

truck_id,total_maintenance_cost,total_downtime_hours,maintenance_events,total_miles_driven,average_miles_per_trip,maintenance_cost_per_mile,downtime_hours_per_1000_miles
TRK00017,46265.84,510.0,20,null,null,null,null
TRK00076,30133.21,388.9,17,1280075,1409.77,0.02,0.3
TRK00052,63757.21,737.3,31,1241107,1392.94,0.05,0.59
TRK00070,36142.32,307.7,15,1319519,1399.28,0.03,0.23
TRK00117,25579.03,271.0,15,null,null,null,null
TRK00025,44082.12,570.1,23,1370297,1422.95,0.03,0.42
TRK00027,44230.56,599.3,24,null,null,null,null
TRK00098,41318.95,460.3,18,1307617,1435.36,0.03,0.35
TRK00064,46296.36,510.2,22,1296870,1412.71,0.04,0.39
TRK00080,44894.62,504.4,21,1268474,1417.29,0.04,0.4


**5. **Fuel Efficiency:** How does fuel efficiency (MPG) trend over time, and which routes have the highest and lowest fuel costs?**


In [0]:
# ============================================================
# BUSINESS QUESTION 5: FUEL EFFICIENCY
# How does fuel efficiency trend over time, and which routes
# have highest/lowest fuel costs?
# ============================================================

# Calculate monthly fuel efficiency from trip data
fuel_efficiency_trend = (
    trips
    .withColumn("year", year("dispatch_date"))
    .withColumn("month", month("dispatch_date"))
    .groupBy(
        "year",
        "month"
    )
    .agg(
        round(avg("average_mpg"), 2).alias("average_mpg"),
        round(sum("fuel_gallons_used"), 2).alias("total_fuel_gallons"),
        sum("actual_distance_miles").alias("total_miles_driven")
    )
    .withColumn(
        "miles_per_gallon",
        round(
            col("total_miles_driven") /
            col("total_fuel_gallons"),
            2
        )
    )
)

# Calculate fuel cost by route
route_fuel_cost = (
    fuel_purchases.alias("f")
    .join(
        trips.alias("t"),
        col("f.trip_id") == col("t.trip_id"),
        "left"
    )
    .join(
        loads.alias("l"),
        col("t.load_id") == col("l.load_id"),
        "left"
    )
    .join(
        routes.alias("r"),
        col("l.route_id") == col("r.route_id"),
        "left"
    )
    .groupBy(
        col("l.route_id").alias("route_id"),
        col("r.origin_city").alias("origin_city"),
        col("r.origin_state").alias("origin_state"),
        col("r.destination_city").alias("destination_city"),
        col("r.destination_state").alias("destination_state")
    )
    .agg(
        round(sum("f.total_cost"), 2).alias("total_fuel_cost"),
        round(sum("f.gallons"), 2).alias("total_fuel_gallons")
    )
    .withColumn(
        "fuel_cost_per_gallon",
        round(
            col("total_fuel_cost") /
            col("total_fuel_gallons"),
            2
        )
    )
)

# Define the Gold table paths
fuel_efficiency_trend_path = f"{gold_base}/fuel_efficiency_trend"
route_fuel_cost_path = f"{gold_base}/route_fuel_cost"

# Create the monthly fuel efficiency Gold table
# on the first run
if not DeltaTable.isDeltaTable(spark, fuel_efficiency_trend_path):

    fuel_efficiency_trend.write \
        .format("delta") \
        .save(fuel_efficiency_trend_path)

# Update existing monthly records and insert new records
else:

    fuel_efficiency_table = DeltaTable.forPath(
        spark,
        fuel_efficiency_trend_path
    )

    (
        fuel_efficiency_table.alias("target")
        .merge(
            fuel_efficiency_trend.alias("source"),
            """
            target.year = source.year
            AND target.month = source.month
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# Create the route fuel cost Gold table on the first run
if not DeltaTable.isDeltaTable(spark, route_fuel_cost_path):

    route_fuel_cost.write \
        .format("delta") \
        .save(route_fuel_cost_path)

# Update existing routes and insert new routes
else:

    route_fuel_cost_table = DeltaTable.forPath(
        spark,
        route_fuel_cost_path
    )

    (
        route_fuel_cost_table.alias("target")
        .merge(
            route_fuel_cost.alias("source"),
            "target.route_id = source.route_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# Display monthly fuel efficiency
display(
    spark.read
    .format("delta")
    .load(fuel_efficiency_trend_path)
    .limit(30)
)

# Display fuel cost by route
display(
    spark.read
    .format("delta")
    .load(route_fuel_cost_path)
    .limit(30)
)

year,month,average_mpg,total_fuel_gallons,total_miles_driven,miles_per_gallon
2022,9,6.49,511054.4,3292513,6.44
2023,10,6.49,538194.1,3467415,6.44
2023,6,6.52,520553.0,3368390,6.47
2024,1,6.48,533532.0,3430157,6.43
2023,1,6.49,541938.0,3489756,6.44
2022,8,6.5,556760.8,3587068,6.44
2024,6,6.51,537550.3,3470010,6.46
2022,2,6.48,478335.6,3077246,6.43
2024,5,6.51,545687.4,3525548,6.46
2022,3,6.5,534854.1,3444911,6.44


route_id,origin_city,origin_state,destination_city,destination_state,total_fuel_cost,total_fuel_gallons,fuel_cost_per_gallon
RTE00044,Charlotte,NC,Portland,OR,1553387.9,398694.3,3.9
RTE00008,Dallas,TX,Denver,CO,1642198.95,420061.9,3.91
RTE00013,Phoenix,AZ,Denver,CO,1613793.72,413922.1,3.9
RTE00027,Detroit,MI,Houston,TX,1694477.33,433072.2,3.91
RTE00004,Chicago,IL,Portland,OR,1618736.89,414884.7,3.9
RTE00055,Kansas City,MO,Miami,FL,1607422.34,412656.3,3.9
RTE00026,Miami,FL,Denver,CO,1566534.68,402244.5,3.89
RTE00025,Miami,FL,Seattle,WA,1641525.7,420348.8,3.91
RTE00017,Philadelphia,PA,Minneapolis,MN,1691548.51,432881.0,3.91
RTE00038,Las Vegas,NV,Kansas City,MO,1593184.71,408525.9,3.9


**6. **Customer Analysis:** How much revenue is generated from each customer, and how well are customer service-level requirements being met?**


In [0]:
# ============================================================
# BUSINESS QUESTION 6: CUSTOMER ANALYSIS
# How much revenue is generated from each customer, and how
# well are customer service-level requirements met?
# ============================================================

# Calculate revenue and load volume for each customer
customer_revenue = (
    loads.alias("l")
    .join(
        customers.alias("c"),
        col("l.customer_id") == col("c.customer_id"),
        "left"
    )
    .groupBy(
        col("l.customer_id").alias("customer_id"),
        col("c.customer_name").alias("customer_name"),
        col("c.customer_type").alias("customer_type")
    )
    .agg(
        count("l.load_id").alias("total_loads"),
        round(sum("l.revenue"), 2).alias("total_revenue")
    )
)

# Calculate on-time delivery performance for each customer
customer_service_level = (
    delivery_events.alias("e")
    .join(
        loads.alias("l"),
        col("e.load_id") == col("l.load_id"),
        "left"
    )
    .groupBy(
        col("l.customer_id").alias("customer_id")
    )
    .agg(
        round(
            avg(
                when(
                    col("e.on_time_flag") == True,
                    1
                ).otherwise(0)
            ) * 100,
            2
        ).alias("on_time_delivery_rate"),
        round(
            avg(col("e.detention_minutes")),
            2
        ).alias("average_detention_minutes")
    )
)

# Combine customer revenue and service-level metrics
customer_analysis_gold = (
    customer_revenue
    .join(
        customer_service_level,
        "customer_id",
        "left"
    )
)

# Define the Gold table path
customer_analysis_gold_path = f"{gold_base}/customer_analysis"

# Create the Gold Delta table on the first run
if not DeltaTable.isDeltaTable(
    spark,
    customer_analysis_gold_path
):

    customer_analysis_gold.write \
        .format("delta") \
        .save(customer_analysis_gold_path)

# Update existing customers and insert new customers
else:

    customer_analysis_table = DeltaTable.forPath(
        spark,
        customer_analysis_gold_path
    )

    (
        customer_analysis_table.alias("target")
        .merge(
            customer_analysis_gold.alias("source"),
            "target.customer_id = source.customer_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# Display the final Customer Analysis Gold table
display(
    spark.read
    .format("delta")
    .load(customer_analysis_gold_path)
    .limit(30)
)

customer_id,customer_name,customer_type,total_loads,total_revenue,on_time_delivery_rate,average_detention_minutes
CUST00175,ABC Wholesale,SPOT,434,1295575.23,55.88,92.56
CUST00166,ABC Distribution,DEDICATED,462,1401500.06,55.95,94.09
CUST00008,American Logistics,CONTRACT,405,1271882.64,56.67,87.36
CUST00005,American Distribution,SPOT,418,1317832.28,54.9,98.74
CUST00021,Elite Industries,DEDICATED,436,1362145.97,54.59,90.74
CUST00190,Global Wholesale,DEDICATED,448,1370567.97,55.25,93.37
CUST00050,Global Manufacturing,SPOT,425,1286487.53,57.41,90.71
CUST00156,First Logistics,DEDICATED,382,1128763.05,57.46,96.43
CUST00019,American Supply Chain,CONTRACT,437,1327655.2,55.03,94.2
CUST00087,Global Logistics,CONTRACT,445,1410955.15,54.94,89.34


**7. **Safety Metrics:** What are the incident rates and number of preventable accidents by driver, vehicle, route, or time period?**


In [0]:
# ============================================================
# BUSINESS QUESTION 7: SAFETY METRICS
# What are incident rates and preventable accidents by driver,
# vehicle, route, or time period?
# ============================================================

# Calculate safety incidents by driver, truck, and route
safety_metrics = (
    safety_incidents.alias("s")
    .join(
        trips.alias("t"),
        col("s.trip_id") == col("t.trip_id"),
        "left"
    )
    .join(
        loads.alias("l"),
        col("t.load_id") == col("l.load_id"),
        "left"
    )
    .join(
        routes.alias("r"),
        col("l.route_id") == col("r.route_id"),
        "left"
    )
    .join(
        drivers.alias("d"),
        col("s.driver_id") == col("d.driver_id"),
        "left"
    )
    .join(
        trucks.alias("tr"),
        col("s.truck_id") == col("tr.truck_id"),
        "left"
    )
    .groupBy(
        col("s.driver_id").alias("driver_id"),
        col("d.first_name").alias("first_name"),
        col("d.last_name").alias("last_name"),
        col("s.truck_id").alias("truck_id"),
        col("tr.unit_number").alias("unit_number"),
        col("l.route_id").alias("route_id"),
        col("r.origin_city").alias("origin_city"),
        col("r.destination_city").alias("destination_city")
    )
    .agg(
        count("s.incident_id").alias("total_incidents"),

        sum(
            when(
                col("s.preventable_flag") == True,
                1
            ).otherwise(0)
        ).alias("preventable_incidents"),

        sum(
            when(
                col("s.at_fault_flag") == True,
                1
            ).otherwise(0)
        ).alias("at_fault_incidents"),

        sum(
            when(
                col("s.injury_flag") == True,
                1
            ).otherwise(0)
        ).alias("injury_incidents"),

        round(
            sum("s.vehicle_damage_cost"),
            2
        ).alias("vehicle_damage_cost"),

        round(
            sum("s.cargo_damage_cost"),
            2
        ).alias("cargo_damage_cost"),

        round(
            sum("s.claim_amount"),
            2
        ).alias("total_claim_amount")
    )
)

# Calculate preventable incident rate
safety_metrics_gold = (
    safety_metrics
    .withColumn(
        "preventable_incident_rate",
        round(
            (
                col("preventable_incidents") /
                col("total_incidents")
            ) * 100,
            2
        )
    )
)

# Define the Gold table path
safety_metrics_gold_path = f"{gold_base}/safety_metrics"

# Create the Gold Delta table on the first run
if not DeltaTable.isDeltaTable(
    spark,
    safety_metrics_gold_path
):

    safety_metrics_gold.write \
        .format("delta") \
        .save(safety_metrics_gold_path)

# Update existing safety records and insert new records
else:

    safety_metrics_table = DeltaTable.forPath(
        spark,
        safety_metrics_gold_path
    )

    (
        safety_metrics_table.alias("target")
        .merge(
            safety_metrics_gold.alias("source"),
            """
            target.driver_id = source.driver_id
            AND target.truck_id = source.truck_id
            AND target.route_id = source.route_id
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# Display the final Safety Metrics Gold table
display(
    spark.read
    .format("delta")
    .load(safety_metrics_gold_path)
    .limit(30)
)

driver_id,first_name,last_name,truck_id,unit_number,route_id,origin_city,destination_city,total_incidents,preventable_incidents,at_fault_incidents,injury_incidents,vehicle_damage_cost,cargo_damage_cost,total_claim_amount,preventable_incident_rate
DRV00006,Robert,Taylor,TRK00084,2502,RTE00040,Minneapolis,Los Angeles,1,0,0,0,2700.7,14284.24,16984.94,0.0
DRV00060,Jessica,Thomas,TRK00115,4533,RTE00048,Columbus,Portland,1,0,0,0,15572.01,0.0,15572.01,0.0
DRV00002,William,Martin,TRK00025,8967,RTE00036,Las Vegas,Los Angeles,1,0,0,1,22117.89,0.0,22117.89,0.0
DRV00045,Thomas,Martinez,TRK00024,6598,RTE00007,Dallas,New York,1,0,0,0,3783.54,0.0,3783.54,0.0
DRV00104,Richard,Hernandez,TRK00062,7256,RTE00035,Las Vegas,Dallas,1,1,1,0,13299.51,0.0,13299.51,100.0
DRV00124,Thomas,Jackson,TRK00093,7126,RTE00009,Dallas,Indianapolis,1,0,1,1,23366.57,0.0,23366.57,0.0
DRV00127,Thomas,Gonzalez,TRK00024,6598,RTE00028,Seattle,Houston,1,0,1,1,24102.19,0.0,24102.19,0.0
DRV00124,Thomas,Jackson,TRK00006,6082,RTE00031,Denver,Kansas City,1,1,1,0,17783.76,0.0,17783.76,100.0
DRV00044,Richard,Davis,TRK00055,6070,RTE00046,Columbus,Los Angeles,1,0,0,0,19373.44,0.0,19373.44,0.0
DRV00018,Patricia,Martin,TRK00074,9452,RTE00043,Charlotte,Denver,1,0,0,0,14176.42,32.85,14209.27,0.0


**8. **Seasonal Patterns:** How do load volumes and freight rates fluctuate across different seasons, months, or periods?**

In [0]:
# ============================================================
# BUSINESS QUESTION 8: SEASONAL PATTERNS
# How do load volumes and freight rates fluctuate across
# seasons/months/periods?
# ============================================================

# Create monthly load and freight-rate metrics
seasonal_metrics = (
    loads.alias("l")
    .join(
        routes.alias("r"),
        col("l.route_id") == col("r.route_id"),
        "left"
    )
    .withColumn(
        "year",
        year(col("l.load_date"))
    )
    .withColumn(
        "month",
        month(col("l.load_date"))
    )
    .withColumn(
        "season",
        when(col("month").isin(12, 1, 2), "Winter")
        .when(col("month").isin(3, 4, 5), "Spring")
        .when(col("month").isin(6, 7, 8), "Summer")
        .when(col("month").isin(9, 10, 11), "Fall")
    )
    .groupBy(
        "year",
        "month",
        "season"
    )
    .agg(
        count("l.load_id").alias("total_loads"),

        round(
            sum("l.revenue"),
            2
        ).alias("total_revenue"),

        round(
            avg("l.revenue"),
            2
        ).alias("average_revenue_per_load"),

        round(
            avg("r.base_rate_per_mile"),
            2
        ).alias("average_base_rate_per_mile"),

        round(
            avg("r.fuel_surcharge_rate"),
            2
        ).alias("average_fuel_surcharge_rate")
    )
)

# Calculate revenue contribution per load
seasonal_metrics_gold = (
    seasonal_metrics
    .withColumn(
        "revenue_per_load",
        round(
            col("total_revenue") /
            col("total_loads"),
            2
        )
    )
)

# Define the Gold table path
seasonal_metrics_gold_path = f"{gold_base}/seasonal_patterns"

# Create the Gold Delta table on the first run
if not DeltaTable.isDeltaTable(
    spark,
    seasonal_metrics_gold_path
):

    seasonal_metrics_gold.write \
        .format("delta") \
        .save(seasonal_metrics_gold_path)

# Update existing monthly records and insert new records
else:

    seasonal_metrics_table = DeltaTable.forPath(
        spark,
        seasonal_metrics_gold_path
    )

    (
        seasonal_metrics_table.alias("target")
        .merge(
            seasonal_metrics_gold.alias("source"),
            """
            target.year = source.year
            AND target.month = source.month
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# Display the final Seasonal Patterns Gold table
display(
    spark.read
    .format("delta")
    .load(seasonal_metrics_gold_path)
    .limit(30)
)

year,month,season,total_loads,total_revenue,average_revenue_per_load,average_base_rate_per_mile,average_fuel_surcharge_rate,revenue_per_load
2024,8,Summer,2426,7349694.49,3029.55,2.21,0.25,3029.55
2022,12,Winter,2375,7341477.11,3091.15,2.2,0.25,3091.15
2023,5,Spring,2348,7243795.22,3085.09,2.2,0.25,3085.09
2022,11,Fall,2360,7260206.36,3076.36,2.21,0.25,3076.36
2024,3,Spring,2431,7453599.0,3066.06,2.2,0.25,3066.06
2022,2,Winter,2179,6574259.06,3017.1,2.18,0.25,3017.1
2023,11,Fall,2364,7235373.47,3060.65,2.19,0.25,3060.65
2023,4,Spring,2336,7289831.14,3120.65,2.2,0.25,3120.65
2023,9,Fall,2327,7018436.2,3016.09,2.19,0.25,3016.09
2022,9,Fall,2304,7108889.83,3085.46,2.2,0.25,3085.46
